<a href="https://colab.research.google.com/github/uzcaliskan/Magibu/blob/main/llm_lora_fine_tuned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

In [ ]:
import json
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from datasets import Dataset
from trl import SFTTrainer, SFTConfig


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
MAX_SEQ_LENGTH = 2048   # veri setindeki p95 ~510, maks ~638 token (thinking dahil, yaklaşık ölçüm) + tampon
LORA_RANK = 16
LORA_ALPHA = 16
BATCH_SIZE = 1
GRAD_ACCUM = 8          # etkin batch = 1*8 = 8
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3.5-4B",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # otomatik seçilsin (bf16 destekleniyorsa o, yoksa fp16)
)

==((====))==  Unsloth 2026.7.5: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers
    r=LORA_RANK,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_rs_lora=True,
    use_gradient_checkpointing="unsloth",  # VRAM tasarrufu için zorunlu
    random_state=42,
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [ ]:
with open("/content/sondaj_finetune_veriseti.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

In [ ]:
dataset = Dataset.from_json("/content/sondaj_finetune_veriseti.json")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['content', 'images', 'role', 'thinking', 'tool_calls'],
    num_rows: 3822
})

In [ ]:
def format_chat_data(batch):
    texts = []
    # Verideki tüm elemanları ikişerli (user ve assistant) alıyoruz
    for i in range(0, len(batch["role"]) - 1, 2):
        if batch["role"][i] == "user" and batch["role"][i+1] == "assistant":
            user_content = batch["content"][i]
            asst_content = batch["content"][i+1]

            messages = [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": asst_content}
            ]

            # Tokenizer'ın sohbet şablonunu uyguluyoruz
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)

    return {"text": texts}

# Veri setini dönüştürüyoruz
formatted_dataset = dataset.map(format_chat_data, batched=True, remove_columns=dataset.column_names)

Map:   0%|          | 0/3822 [00:00<?, ? examples/s]

In [ ]:
# from transformers import TrainingArguments
from unsloth import UnslothTrainer, UnslothTrainingArguments

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    eval_dataset = None,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,


    args = UnslothTrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,

        # Use warmup_ratio and num_train_epochs for longer runs!
        #max_steps = 25,
        warmup_steps = 2,
        # warmup_ratio = 0.1,
        # num_train_epochs = 1,

        # Select a 2 to 10x smaller learning rate for the embedding matrices!
        learning_rate = 5e-5,
        embedding_learning_rate = 1e-6,

        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/content/",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,911 | Num Epochs = 3 | Total steps = 1,434
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.327959
2,1.461281
3,1.776778
4,1.087682
5,1.281240
6,1.585100
7,1.486707
8,1.392290
9,1.418327


KeyboardInterrupt: 

In [ ]:
model.save_pretrained("lora_model")   # sadece adaptör, birkaç yüz MB
tokenizer.save_pretrained("lora_model")
# veya hub'a:
model.push_to_hub("uzcaliskan/magibu-llm-fine_tuned_sondaj", token="")

README.md:   0%|          | 0.00/567 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  28%|##8       | 24.0MB / 85.0MB            

Saved model to https://huggingface.co/uzcaliskan/magibu-llm-fine_tuned_sondaj


In [ ]:
#model.push_to_hub_merged("uzcaliskan/magibu-llm-fine_tuned_sondaj",
     #                    tokenizer, token = "")

Unsloth: Restored added_tokens_decoder metadata in uzcaliskan/magibu-llm-fine_tuned_sondaj/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   1%|          | 31.9MB / 3.99GB            



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [06:17<00:00, 188.70s/it]


Unsloth: Merge process complete. Saved to `/content/uzcaliskan/magibu-llm-fine_tuned_sondaj`


In [ ]:
!ollama serve

/bin/bash: line 1: ollama: command not found


In [ ]:
dataset.push_to_hub("uzcaliskan/magibu_dataset_drilling", )

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  248kB /  248kB            

CommitInfo(commit_url='https://huggingface.co/datasets/uzcaliskan/magibu_dataset_drilling/commit/44b60ad3c0589b1d8272f64175681b5163bccea5', commit_message='Upload dataset', commit_description='', oid='44b60ad3c0589b1d8272f64175681b5163bccea5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/uzcaliskan/magibu_dataset_drilling', endpoint='https://huggingface.co', repo_type='dataset', repo_id='uzcaliskan/magibu_dataset_drilling'), pr_revision=None, pr_num=None)